[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/60_prefix_cache_solution.ipynb)

# 🟡 Solution: Prefix Cache

Reference solution for `prefix_cache`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

class PrefixCache:
    def __init__(self):
        self.cache = {}

    def add(self, prompt_tokens, key: torch.Tensor, value: torch.Tensor):
        tokens = tuple(int(t) for t in prompt_tokens)
        if key.shape[0] != len(tokens) or value.shape[0] != len(tokens):
            raise ValueError("key/value length must match prompt length")
        self.cache[tokens] = (key.clone(), value.clone())

    def lookup(self, prompt_tokens):
        tokens = tuple(int(t) for t in prompt_tokens)
        best = None
        for cached_tokens, kv in self.cache.items():
            if len(cached_tokens) <= len(tokens) and tokens[:len(cached_tokens)] == cached_tokens:
                if best is None or len(cached_tokens) > len(best[0]):
                    best = (cached_tokens, kv)
        if best is None:
            return 0, None, None
        cached_tokens, (key, value) = best
        return len(cached_tokens), key.clone(), value.clone()


In [ ]:
# Verify
cache = PrefixCache()
cache.add([1, 2, 3], torch.randn(3, 2), torch.randn(3, 2))
print(cache.lookup([1, 2, 3, 4])[0])


In [ ]:
# Run judge
from torch_judge import check
check('prefix_cache')
